In [ ]:
%%sql -r dataframe_1
-- =====================================================
-- PHASE 2B: RAW REDEMPTION LANDING TABLE
-- NOTEBOOK: NB_02_REDEMPTION_INGESTION
-- =====================================================

USE WAREHOUSE SKYPOINTS_WH;
USE DATABASE SKYPOINTS_INCU;
USE SCHEMA RAW;

-- Store original JSON redemption records and file metadata
CREATE TABLE IF NOT EXISTS RAW_REDEMPTIONS (
    RAW_RECORD             VARIANT,
    SOURCE_FILE_NAME       VARCHAR(500) NOT NULL,
    FILE_TIMESTAMP         TIMESTAMP_NTZ,
    INGESTION_TIMESTAMP    TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
    BATCH_ID               VARCHAR(100) NOT NULL,
    SOURCE_RECORD_NUMBER   NUMBER(38,0)
);

-- Verify RAW tables and stages
DESCRIBE TABLE RAW_REDEMPTIONS;

SHOW TABLES IN SCHEMA RAW;

SHOW STAGES IN SCHEMA RAW;

In [ ]:
-- NB_02_REDEMPTION_INGESTION
-- CELL 2: VERIFY REDEMPTION FILE

USE DATABASE SKYPOINTS_INCU;

LIST @RAW.REDEMPTION_STAGE;

In [ ]:
-- CELL 3: LOAD REDEMPTION JSON INTO RAW

USE WAREHOUSE SKYPOINTS_WH;
USE DATABASE SKYPOINTS_INCU;

COPY INTO RAW.RAW_REDEMPTIONS
(
    RAW_RECORD,
    SOURCE_FILE_NAME,
    FILE_TIMESTAMP,
    INGESTION_TIMESTAMP,
    BATCH_ID,
    SOURCE_RECORD_NUMBER
)
FROM (
    SELECT
        $1,
        METADATA$FILENAME,
        TO_TIMESTAMP_NTZ(
            REGEXP_SUBSTR(
                METADATA$FILENAME,
                '[0-9]{8}_[0-9]{6}'
            ),
            'YYYYMMDD_HH24MISS'
        ),
        CURRENT_TIMESTAMP()::TIMESTAMP_NTZ,
        MD5(
            METADATA$FILENAME || '|' ||
            REGEXP_SUBSTR(
                METADATA$FILENAME,
                '[0-9]{8}_[0-9]{6}'
            )
        ),
        METADATA$FILE_ROW_NUMBER
    FROM @RAW.REDEMPTION_STAGE
)
FILES = ('REDEMPTION_20261125_202700.json')
FILE_FORMAT = (
    TYPE = JSON
)
ON_ERROR = 'ABORT_STATEMENT';

In [ ]:
SELECT
    RAW_RECORD,
    SOURCE_FILE_NAME,
    FILE_TIMESTAMP,
    INGESTION_TIMESTAMP,
    BATCH_ID,
    SOURCE_RECORD_NUMBER
FROM RAW.RAW_REDEMPTIONS
WHERE SOURCE_FILE_NAME LIKE '%REDEMPTION_20261125_202700.json%';